# DS2002 · Analysis Clinic with SQL in SQLite

**Lecture — 2026-11-30 · Fall 2026**  
**Class time:** 45 minutes

---

## Analysis clinic: SQL in SQLite

The capstone asks you to answer every question with a query against your own database. Most teams know enough SQL to select and group, so this clinic covers the four patterns that the capstone questions need and that we have used less often: turning a games table into one row per team, joining each game to the ratings that existed before it, computing values across a team's recent games with a window function, and loading data so that a second run does not create duplicates.

Every example uses a made-up league of four teams called Harbor, Ridge, Summit, and Valley. The league is small on purpose, so you can check each result by eye. None of these queries will run on your capstone database without changes, and writing those changes is the work of the next week.

Bring your team notebook, because the last twenty minutes are for questions about your own data.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE games (
    game_id INTEGER PRIMARY KEY, game_date TEXT,
    home TEXT, away TEXT, home_pts INTEGER, away_pts INTEGER);
INSERT INTO games VALUES
    (1, '2026-11-02', 'Harbor', 'Ridge',  81, 64),
    (2, '2026-11-04', 'Summit', 'Valley', 70, 72),
    (3, '2026-11-06', 'Ridge',  'Summit', 66, 61),
    (4, '2026-11-08', 'Valley', 'Harbor', 59, 77),
    (5, '2026-11-10', 'Harbor', 'Summit', 68, 66),
    (6, '2026-11-12', 'Ridge',  'Valley', 74, 73),
    (7, '2026-11-14', 'Summit', 'Harbor', 80, 71),
    (8, '2026-11-16', 'Valley', 'Ridge',  62, 75);

CREATE TABLE ratings (snapshot_date TEXT, team TEXT, rating REAL,
                      PRIMARY KEY (snapshot_date, team));
INSERT INTO ratings VALUES
    ('2026-11-01', 'Harbor', 0.82), ('2026-11-01', 'Ridge', 0.55),
    ('2026-11-01', 'Summit', 0.61), ('2026-11-01', 'Valley', 0.40),
    ('2026-11-09', 'Harbor', 0.85), ('2026-11-09', 'Ridge', 0.63),
    ('2026-11-09', 'Summit', 0.52), ('2026-11-09', 'Valley', 0.44);
""")
print('SQLite version:', sqlite3.sqlite_version, '(window functions need 3.25 or newer)')
pd.read_sql_query('SELECT * FROM games', conn)

### Pattern 1: one row per team per game

The `games` table stores each game once, with a home team and an away team. Questions about a single team, such as its record or its average margin, are much easier when every game appears twice, once from each team's point of view. A common table expression (the `WITH` clause) builds that view without storing a second table, and `UNION ALL` stacks the home rows on top of the away rows.

In [ ]:
team_view = '''
WITH team_games AS (
    SELECT game_id, game_date, home AS team, away AS opponent,
           home_pts AS pts_for, away_pts AS pts_against
    FROM games
    UNION ALL
    SELECT game_id, game_date, away, home, away_pts, home_pts
    FROM games
)
'''

pd.read_sql_query(team_view + '''
SELECT team, COUNT(*) AS games,
       SUM(pts_for > pts_against) AS wins,
       ROUND(AVG(pts_for - pts_against), 1) AS avg_margin
FROM team_games
GROUP BY team
ORDER BY avg_margin DESC
''', conn)

Two details are worth noticing. In SQLite a comparison such as `pts_for > pts_against` evaluates to 1 or 0, so `SUM` of it counts the wins. The second half of the `UNION ALL` does not repeat the column names, because SQLite takes them from the first half, which is why the order of the columns has to match.

**Try it:** add a column that counts close wins, meaning wins by five points or fewer. If the query is right, Harbor shows exactly one.

In [ ]:
# TODO: copy the query above and add a close_wins column

### Pattern 2: the rating that existed before the game

Ratings change as the season goes on. If you want to know whether the better-rated team won a game, you have to use a rating published before the game, or the rating already includes the result you are trying to predict. Analysts call that mistake look-ahead.

A correlated subquery finds, for each game, the latest snapshot dated before the game.

In [ ]:
pd.read_sql_query('''
SELECT g.game_id, g.game_date, g.home, g.away,
       (SELECT MAX(snapshot_date) FROM ratings
        WHERE snapshot_date < g.game_date) AS snapshot_used
FROM games g
''', conn)

Once you know which snapshot applies, join it twice, once for each team, and compare. Aliases (`rh` for the home team's rating and `ra` for the away team's) keep the two joins apart.

In [ ]:
pd.read_sql_query('''
WITH g AS (
    SELECT games.*,
           (SELECT MAX(snapshot_date) FROM ratings
            WHERE snapshot_date < games.game_date) AS snap
    FROM games
)
SELECT g.game_id, g.home, rh.rating AS home_rating,
       g.away, ra.rating AS away_rating,
       CASE WHEN rh.rating > ra.rating THEN g.home ELSE g.away END AS favorite,
       CASE WHEN g.home_pts > g.away_pts THEN g.home ELSE g.away END AS winner
FROM g
JOIN ratings rh ON rh.team = g.home AND rh.snapshot_date = g.snap
JOIN ratings ra ON ra.team = g.away AND ra.snapshot_date = g.snap
''', conn)

**Try it:** wrap that query and compute the share of games the favorite won, along with the number of games. A share reported without its count is hard to judge, because three out of four and seventy-five out of one hundred are not equally convincing.

In [ ]:
# TODO: favorite win rate and n

### Pattern 3: window functions for recent form

An aggregate such as `AVG` collapses many rows into one. A window function computes the same kind of value but keeps every row, which is what you want for a rolling average across a team's recent games. The `OVER` clause says which rows belong in the window: `PARTITION BY` starts over for each team, `ORDER BY` puts the games in date order, and the `ROWS BETWEEN` phrase limits the window to the current game and the ones before it.

In [ ]:
pd.read_sql_query(team_view + '''
SELECT team, game_date, pts_for - pts_against AS margin,
       ROUND(AVG(pts_for - pts_against) OVER (
           PARTITION BY team ORDER BY game_date
           ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS rolling_3
FROM team_games
ORDER BY team, game_date
''', conn)

The first two rows for each team average fewer than three games, because there are not three games yet. Decide whether your chart should show those early values or start once the window is full, and say which choice you made.

`ROW_NUMBER()` is another window function you will use. It numbers the rows inside each partition, which makes it easy to keep only the top row for each team. The toy league has no players, so this example ranks each team's games by margin and keeps the best one.

In [ ]:
pd.read_sql_query(team_view + '''
, ranked AS (
    SELECT team, game_date, opponent, pts_for - pts_against AS margin,
           ROW_NUMBER() OVER (PARTITION BY team
                              ORDER BY pts_for - pts_against DESC) AS rn
    FROM team_games
)
SELECT team, game_date, opponent, margin
FROM ranked
WHERE rn = 1
''', conn)

**Try it:** change the window in the rolling query to the last five games and confirm that the first four rows for each team match a plain running average.

In [ ]:
# TODO

### Pattern 4: loading so a rerun is safe

Your pipeline will run many times before December 18. If the load uses a plain `INSERT`, every run adds another copy of each game, and averages drift without any error message. A primary key plus `INSERT OR REPLACE` makes the load safe to repeat: when a row with the same key already exists, SQLite replaces it instead of adding a second one.

In [ ]:
conn.execute('CREATE TABLE scores (game_id INTEGER, team TEXT, pts INTEGER)')
conn.execute('CREATE TABLE scores_keyed (game_id INTEGER, team TEXT, pts INTEGER, '
             'PRIMARY KEY (game_id, team))')

rows = [(1, 'Harbor', 81), (1, 'Ridge', 64)]
for run in range(3):
    conn.executemany('INSERT INTO scores VALUES (?, ?, ?)', rows)
    conn.executemany('INSERT OR REPLACE INTO scores_keyed VALUES (?, ?, ?)', rows)

for table in ['scores', 'scores_keyed']:
    n = conn.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:13s} rows after three runs: {n}')

The table without a key holds six rows after three runs, so every total computed from it would be three times too large. The keyed table still holds two.

### From a query to a chart

`pd.read_sql_query` returns a DataFrame, so the charting you learned in the visualization unit picks up from there. Compute the number in the title from the query result, so the title and the chart cannot disagree.

In [ ]:
import matplotlib.pyplot as plt

summary = pd.read_sql_query(team_view + '''
SELECT team, ROUND(AVG(pts_for - pts_against), 1) AS avg_margin
FROM team_games GROUP BY team ORDER BY avg_margin
''', conn)

best = summary.iloc[-1]
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(summary['team'], summary['avg_margin'])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title(f"{best['team']} leads the toy league with a {best['avg_margin']:+.1f} "
             'average margin', loc='left')
ax.set_xlabel('average margin (points)')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

### For the rest of the hour

Open your team notebook and pick the capstone question that is furthest behind. Most teams find Question 6 the hardest because it combines patterns 2 and 4, so if your snapshots are not loading cleanly, start there. Wednesday's studio is about publishing your API, which reads from the same database, so the tables need to be in good shape by then.